In [1]:
import surprise
import pandas as pd
import numpy as np


import os
os.chdir(r'E:\PGCP-AI\PML\Cases\SURPRISE\filmtrust') 

In [2]:
ratings = pd.read_csv('ratings.txt' , sep=' ', names= ['uid','iid','rating'])
ratings.head()   

,uid,iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0


In [3]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings['rating'].max()
lowest_rating,highest_rating

(np.float64(0.5), np.float64(4.0))

In [4]:
reader =surprise.Reader(rating_scale=(lowest_rating, highest_rating))

data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [5]:
similarity_options = {'name':'cosine', 'user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())
# calculate the expected ratings for all the users

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [6]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold 

In [7]:
param_grid = {
    'n_epochs': [50],
    'n_factors':[100,50],
    'lr_all': np.linspace(0.001,1,5),
    'reg_all': np.linspace(0.01, 0.8, 5)
}
kfold = KFold(n_splits = 5, random_state = 26, shuffle = True)

gs = GridSearchCV(surprise.SVD , param_grid , measures = ['rmse','mae'], joblib_verbose = 3, cv = kfold , n_jobs = -1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:   12.1s
[Parallel(n_jobs=-1)]: Done 250 out of 250 | elapsed:   24.8s finished


In [8]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8078874250859462
{'n_epochs': 50, 'n_factors': 50, 'lr_all': np.float64(0.001), 'reg_all': np.float64(0.01)}


In [9]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())